In [1]:
import os

In [2]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project'

In [5]:
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class DataTransformationconfig:
    root_dir:Path
    data_path:Path

In [6]:
from WineQuality_Project.utils.common import read_yaml,create_directories
from WineQuality_Project.constants import *

In [7]:
class ConfigManager:
    def __init__(self,
                 config_filepath: Path = CONFIG_FILE_PATH,
                 params_filepath: Path = PARAMS_FILE_PATH,
                 schema_filepath: Path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([Path(self.config['artifact_root'])])

    def get_data_transformation_config(self) -> DataTransformationconfig:
        config = self.config["data_transformation"]   # ← FIXED (dict access)

        root_dir = Path(config["root_dir"])
        data_path = Path(config["data_path"])

        create_directories([root_dir])  # ← FIXED

        data_transformation_config = DataTransformationconfig(
            root_dir=root_dir,
            data_path=data_path
        )

        return data_transformation_config


In [8]:
import os
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
import logging

class DataTransformation:
    def __init__(self, config: DataTransformationconfig):
        self.config = config
        self.config.root_dir.mkdir(parents=True, exist_ok=True)

    def initiate_data_transformation(self):
        try:
            # 1️⃣ Load the validated data
            df = pd.read_csv(self.config.data_path)
            logging.info(f"Loaded data: {self.config.data_path}")

            # 2️⃣ Split into train and test
            train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
            logging.info("Split data into train and test")

            # 3️⃣ Save the train and test CSVs
            train_path = self.config.root_dir / "train.csv"
            test_path = self.config.root_dir / "test.csv"
            train_df.to_csv(train_path, index=False)
            test_df.to_csv(test_path, index=False)

            logging.info(f"Saved train.csv at: {train_path}")
            logging.info(f"Saved test.csv at: {test_path}")

            # 4️⃣ Return paths for the next stage
            return train_path, test_path

        except Exception as e:
            logging.error(f"Data Transformation failed: {e}")
            raise e


In [12]:
try:
    config = ConfigManager()
    
    data_transformation_config = config.get_data_transformation_config()
    
    data_transformation = DataTransformation(config=data_transformation_config)
    
    artifacts = data_transformation.initiate_data_transformation()

    print("✔ Data Transformation Completed Successfully!")
    print(artifacts)

except Exception as e:
    print(" Error in Data Transformation Stage")
    raise e


[2025-12-12 05:00:07] [INFO] WineQualityLogger - YAML file: config\config.yml loaded successfully
INFO:WineQualityLogger:YAML file: config\config.yml loaded successfully
[2025-12-12 05:00:07] [INFO] WineQualityLogger - YAML file: params.yaml loaded successfully
INFO:WineQualityLogger:YAML file: params.yaml loaded successfully
[2025-12-12 05:00:07] [INFO] WineQualityLogger - YAML file: schema.yaml loaded successfully
INFO:WineQualityLogger:YAML file: schema.yaml loaded successfully
[2025-12-12 05:00:07] [INFO] WineQualityLogger - Directory created at: artifacts
INFO:WineQualityLogger:Directory created at: artifacts
[2025-12-12 05:00:07] [INFO] WineQualityLogger - Directory created at: artifacts\data_transformation
INFO:WineQualityLogger:Directory created at: artifacts\data_transformation


✔ Data Transformation Completed Successfully!
(WindowsPath('artifacts/data_transformation/train.csv'), WindowsPath('artifacts/data_transformation/test.csv'))
